In [1]:
import sys, time
from pprint import pprint
# time.sleep(10)

pprint(sys.version)
from pathlib import Path

HOME_DIR = Path.home()
CODE_DIR = HOME_DIR / 'synthesizrr' / 'src'
sys.path.insert(-2, '/home/ec2-user/anaconda3/envs/hft4/lib/python3.11/site-packages')
sys.path.insert(-2, '/home/ec2-user/anaconda3/envs/hft4/bin/')
sys.path.insert(-2, str(CODE_DIR))
pprint(sys.path)

import synthergent
from typing import *
import time, glob, os, sys, boto3, numpy as np, pandas as pd, json, requests, gc
from pandas.core.frame import DataFrame as PandasDataFrame, Series as PandasSeries
from pathlib import Path

RAY_TMP_DIR = '/tmp/ray/'

from synthergent.base.util import *
from synthergent.base.data import *
from synthergent.base.constants import *
from synthergent.base.framework import *
from synthergent.base.data.reader import Reader
from synthergent.base.data.writer import Writer
from synthergent.base.framework.dl.torch import *
from synthergent.base.framework.task_data import DataSplit, Datasets, TaskData
import synthergent.base.algorithm
import synthergent.base.metric
from synthergent.base.framework.task.classification import _normalize_label
import datasets as ds
# from datasets import load_dataset, load_from_disk
from synthergent.base.framework.trainer.RayTuneTrainer import _ray_metric_str
from termcolor import COLORS as TERMCOLOR_COLORS
from termcolor import colored
import ray
from ray.util.dask import ray_dask_get, enable_dask_on_ray, disable_dask_on_ray

from pprint import pprint

os.environ['CUDA_VISIBLE_DEVICES'] = '4,6'

## print = Tracker.default().info

# import hvplot.pandas
# import holoviews as hv
# import plotly.express as px
# import plotly.io as pio
# from IPython.display import display
# from bokeh.palettes import Spectral, Set2, Set3

# pio.templates.default = 'plotly_white'
# hvplot.extension('plotly')
# import numpy as np
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.io as pio
# # pio.renderers.default='iframe'
# # hvplot.extension('bokeh')

# from bokeh.resources import INLINE as BOKEH_INLINE
# from bokeh.io import output_notebook as bokeh_output_notebook
# bokeh_output_notebook(BOKEH_INLINE)


'3.11.8 | packaged by conda-forge | (main, Feb 16 2024, 20:53:32) [GCC 12.3.0]'
['/opt/conda/envs/hft4/lib/python311.zip',
 '/opt/conda/envs/hft4/lib/python3.11',
 '/opt/conda/envs/hft4/lib/python3.11/lib-dynload',
 '/home/ec2-user/anaconda3/envs/hft4/lib/python3.11/site-packages',
 '/home/ec2-user/anaconda3/envs/hft4/bin/',
 '/efs/litmus-server/users/adivekar/synthesizrr/src',
 '',
 '/opt/conda/envs/hft4/lib/python3.11/site-packages']


In [2]:
from datasets import load_dataset

clash_eval = load_dataset('kewu93/ClashEval', split='test', trust_remote_code=True).to_pandas()
clash_eval

,question,context_original,context_mod,answer_original,answer_mod,mod_degree,dataset
0,"Which former United States Senator, born in 19...",Joe Donnelly (born 1955) is a former United St...,Joey McJoeFace (born 1955) is a former United ...,Joe Donnelly,Joey McJoeFace,3,names
1,Who won the Best Actress award at the 1998 Gol...,The 1998 Golden Globes (Portugal) were the thi...,The 1998 Golden Globes (Portugal) were the thi...,Ana Zanatti,Ana Santos,1,names
2,Who is the Canadian actor known for playing Pe...,"Daniel Cudmore (born January 20, 1981) is a Ca...","David Cudmore (born January 20, 1981) is a Can...",Daniel Cudmore,David Cudmore,1,names
3,Who won the Best Actress award at the 1998 Gol...,The 1998 Golden Globes (Portugal) were the thi...,The 1998 Golden Globes (Portugal) were the thi...,Ana Zanatti,Sophia Loren,2,names
4,What was the name of the Officer Commanding th...,"The 167th (Canadien-Français) Battalion, CEF w...","The 167th (Canadien-Français) Battalion, CEF w...",O. Readman,O. Reed'n'Writegood,3,names
...,...,...,...,...,...,...,...
10056,What is the Olympic record for Women's Team pu...,\n\n\n\nList of Olympic records in speed skati...,\n\n\n\nList of Olympic records in speed skati...,2:53.44,4:19.566,1.5,records
10057,What is the Olympic record for Women's Team pu...,\n\n\n\nList of Olympic records in speed skati...,\n\n\n\nList of Olympic records in speed skati...,2:53.44,5:46.88,2,records
10058,What is the Olympic record for Women's Team pu...,\n\n\n\nList of Olympic records in speed skati...,\n\n\n\nList of Olympic records in speed skati...,2:53.44,8:39.132,3,records
10059,What is the Olympic record for Women's Team pu...,\n\n\n\nList of Olympic records in speed skati...,\n\n\n\nList of Olympic records in speed skati...,2:53.44,14:25.22,5,records


In [3]:
import synthergent
from synthergent import Synthergent, Cleaner, QualityCheck, FinalStep
from synthergent.cleaner import Sample

# Calculate LexicalDiversity using 100 rows sample from the dataset

In [4]:
check_modified_context_quality =  Synthergent.of(
    Sample.of(params=dict(size=100, seed=42)),
    FinalStep.of(
        QualityCheck.of('LexicalDiversity', params=dict(col='context_original')),
        QualityCheck.of('LexicalDiversity', params=dict(col='context_mod')),
    ),
)

In [5]:
results = check_modified_context_quality(
    data=clash_eval,
)

Synthergent:   0%|          | 0/2 [00:00<?, ?step/s]

FinalStep:   0%|          | 0/2 [00:00<?, ?step/s]

In [6]:
results['quality_checks'][
    "LexicalDiversity(col='context_original',ngrams=(1,2,3,4,5),spacy_tokenization_model='en_core_web_lg')"
]

,ngram,Self-BLEU
0,1,0.834500
1,2,0.651496
2,3,0.503042
3,4,0.418693
4,5,0.372535


In [8]:
results['quality_checks'][
    "LexicalDiversity(col='context_mod',ngrams=(1,2,3,4,5),spacy_tokenization_model='en_core_web_lg')"
]

,ngram,Self-BLEU
0,1,0.833514
1,2,0.650015
2,3,0.501230
3,4,0.416895
4,5,0.370664


# Make LexicalDiversity calculation 10x faster using Synthergent (processes)

In [9]:
check_modified_context_quality =  Synthergent.of(
    Sample.of(params=dict(size=100, seed=42)),
    FinalStep.of(
        QualityCheck.of('LexicalDiversity', params=dict(col='context_original', num_cpus=20)),
        QualityCheck.of('LexicalDiversity', params=dict(col='context_mod', num_cpus=20)),
    ),
)

In [10]:
results = check_modified_context_quality(
    data=clash_eval,
    scaling=dict(
        parallelize='processes',
        max_workers=40,
    ),
)

Synthergent:   0%|          | 0/2 [00:00<?, ?step/s]

FinalStep:   0%|          | 0/2 [00:00<?, ?step/s]

In [ ]:
print(f'Doing 1000 rows using 1 CPU would require time: {17 * ((1000/10)**2) / 3600:.1f} hours')
print(f'Doing 1000 rows using Synthergent (80 processes): {105.8 * 11 / 3600:.1f} hours')

# Scale LexicalDiversity 10x using Synthergent (Ray)

In [4]:
import ray, dask
from ray.util.dask import enable_dask_on_ray
ray.shutdown()
pprint(ray.init(
    # address='ray://10.0.152.54:10001',
    address='ray://10.0.145.189:10001',
    ignore_reinit_error=True,
    _temp_dir=str(RAY_TMP_DIR),
    runtime_env={"py_modules": [
        synthergent,
    ]},
))
enable_dask_on_ray()
pprint(ray.cluster_resources())

2024-12-16 10:27:49,444	INFO client_builder.py:243 -- Passing the following kwargs to ray.init() on the server: ignore_reinit_error
I0000 00:00:1734344869.462122   49286 config.cc:230] gRPC experiments enabled: call_status_override_on_cancellation, event_engine_dns, event_engine_listener, http2_stats_fix, monitoring_experiment, pick_first_new, trace_record_callops, work_serializer_clears_time_cache
2024-12-16 10:27:50,010	INFO packaging.py:530 -- Creating a file package for local directory '/efs/litmus-server/users/adivekar/synthesizrr/src/synthergent'.
2024-12-16 10:27:50,378	INFO packaging.py:358 -- Pushing file package 'gcs://_ray_pkg_3b52ed99b7bf4f2b.zip' (4.37MiB) to Ray cluster...
2024-12-16 10:27:50,426	INFO packaging.py:371 -- Successfully pushed file package 'gcs://_ray_pkg_3b52ed99b7bf4f2b.zip'.
SIGTERM handler is not set because current thread is not the main thread.


ClientContext(dashboard_url='127.0.0.1:8265',
              python_version='3.11.8',
              ray_version='2.9.2',
              ray_commit='fce7a361807580953364e2da964f9498f3123bf9',
              protocol_version='2023-06-27',
              _num_clients=1,
              _context_to_restore=<ray.util.client._ClientContext object at 0x7efbbcaeea90>)
{'CPU': 96.0,
 'memory': 512356412416.0,
 'node:10.0.145.189': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 268435456000.0}


/opt/conda/envs/hft4/lib/python3.11/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(
(_run_parallel_ray_executor pid=3117) /opt/conda/envs/hft4/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
(_run_parallel_ray_executor pid=3117)   _torch_pytree._register_pytree_node(
(_run_parallel_ray_executor pid=3117) /opt/conda/envs/hft4/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
(_run_parallel_ray_executor pid=3117)   _torch_pytree._register_pytree_node(
(_run_parallel_ray_executor pid=3117) /opt/conda/envs/hft4/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarnin

In [5]:
check_modified_context_quality =  Synthergent.of(
    Sample.of(params=dict(size=100, seed=42)),
    FinalStep.of(
        QualityCheck.of('LexicalDiversity', params=dict(col='context_original', num_cpus=5)),
        QualityCheck.of('LexicalDiversity', params=dict(col='context_mod', num_cpus=5)),
        parallelize='sync',
    ),
)

In [6]:
results = check_modified_context_quality(
    data=clash_eval,
    scaling=dict(
        parallelize='ray',
        max_workers=10,
    ),
)

Synthergent:   0%|          | 0/2 [00:00<?, ?step/s]

/opt/conda/envs/hft4/lib/python3.11/site-packages/ray/util/client/worker.py:614: UserWarning: More than 10MB of messages have been created to schedule tasks on the server. This can be slow on Ray Client due to communication overhead over the network. If you're running many fine-grained tasks, consider running them inside a single remote function. See the section on "Too fine-grained tasks" in the Ray Design Patterns document for more details: https://docs.google.com/document/d/167rnnDFIVRhHhK4mznEIemOtj63IOhtIPvSYaPgI4Fg/edit#heading=h.f7ins22n6nyl. If your functions frequently use large objects, consider storing the objects remotely with ray.put. An example of this is shown in the "Closure capture of large / unserializable object" section of the Ray Design Patterns document, available here: https://docs.google.com/document/d/167rnnDFIVRhHhK4mznEIemOtj63IOhtIPvSYaPgI4Fg/edit#heading=h.1afmymq455wu
  warnings.warn(


FinalStep:   0%|          | 0/2 [00:00<?, ?step/s]

(spacy_tokenize_docs) Started at 2024-12-16T10:28:02.023690+00:00...
(spacy_tokenize_docs) ...completed in 3.16 milliseconds.
(self_bleu_ngram=1) Started at 2024-12-16T10:28:02.027091+00:00...
(self_bleu_ngram=1) ...completed in 46.27 seconds.
(self_bleu_ngram=2) Started at 2024-12-16T10:28:48.298517+00:00...
(self_bleu_ngram=2) ...completed in 11.11 seconds.
(self_bleu_ngram=3) Started at 2024-12-16T10:28:59.409649+00:00...
(self_bleu_ngram=3) ...completed in 11.11 seconds.
(self_bleu_ngram=4) Started at 2024-12-16T10:29:10.523647+00:00...
(self_bleu_ngram=4) ...completed in 11.11 seconds.
(self_bleu_ngram=5) Started at 2024-12-16T10:29:21.636780+00:00...
(self_bleu_ngram=5) ...completed in 15.13 seconds.
(spacy_tokenize_docs) Started at 2024-12-16T10:29:37.110040+00:00...
(spacy_tokenize_docs) ...completed in 3.59 milliseconds.
(self_bleu_ngram=1) Started at 2024-12-16T10:29:37.113843+00:00...
(self_bleu_ngram=1) ...completed in 27.89 seconds.
Error in FinalStep:
[ERROR]: RuntimeErro

TypeError: argument of type 'NoneType' is not iterable